# Reward Model Training

Train một regression model dự đoán engagement score từ text bài đăng Facebook.

**Input:** `dataset/500 posts diningreview.csv`  
**Output:** `reward_model/` — checkpoint dùng làm reward_fn cho GRPO

**Pipeline:**
1. Tính `engage` score từ các cột reaction/like/share/comment
2. Normalize về `[0, 1]`
3. Fine-tune `xlm-roberta-base` (hoặc `vinai/phobert-base`) làm regression head
4. Lưu model + tokenizer vào `REWARD_MODEL_DIR`

## 0. Cài đặt

In [ ]:
# %pip install -U transformers datasets scikit-learn pandas numpy torch accelerate

## 1. Config

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().resolve()
CSV_PATH  = REPO_ROOT / "dataset" / "500 posts diningreview.csv"

# xlm-roberta-base: đa ngôn ngữ, ổn định
# vinai/phobert-base: tiếng Việt tốt hơn nhưng cần tokenizer riêng
BACKBONE_ID = "xlm-roberta-base"

REWARD_MODEL_DIR = REPO_ROOT / "mcs_train_content_model_outputs" / "reward_model"
REWARD_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-5
VAL_RATIO  = 0.15

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device          :", device)
print("backbone        :", BACKBONE_ID)
print("reward_model_dir:", REWARD_MODEL_DIR)

## 2. Load & tính engagement score

In [ ]:
df = pd.read_csv(str(CSV_PATH), encoding="utf-8")

num_cols = [
    "commentsCount", "likesCount", "sharesCount",
    "reactionAngryCount", "reactionCareCount", "reactionHahaCount",
    "reactionLikeCount", "reactionLoveCount", "reactionSadCount", "reactionWowCount",
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["text"] = df["text"].astype(str)
df = df[df["text"].str.len() > 50].copy()

df["engage_raw"] = (
    df["likesCount"]
    + 0.5  * df["commentsCount"]
    + 1.5  * df["sharesCount"]
    + df["reactionLikeCount"]
    + 1.2  * df["reactionLoveCount"]
    + 0.8  * df["reactionHahaCount"]
    + 0.8  * df["reactionCareCount"]
    + 0.8  * df["reactionWowCount"]
    - 0.5  * df["reactionAngryCount"]
    - 0.3  * df["reactionSadCount"]
)
df["engage_log"]  = np.log1p(np.maximum(df["engage_raw"], 0))
df["engage_norm"] = (
    (df["engage_log"] - df["engage_log"].min())
    / (df["engage_log"].max() - df["engage_log"].min())
).astype(float)

print("rows:", len(df))
df[["engage_raw", "engage_log", "engage_norm"]].describe()

## 3. Train / val split & tokenize

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BACKBONE_ID)

train_df, val_df = train_test_split(
    df[["text", "engage_norm"]],
    test_size=VAL_RATIO,
    random_state=SEED,
)

def make_dataset(frame):
    ds = Dataset.from_pandas(frame.reset_index(drop=True))
    ds = ds.map(
        lambda b: tokenizer(
            b["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
        ),
        batched=True,
    )
    ds = ds.rename_column("engage_norm", "labels")
    ds = ds.remove_columns(["text"])
    ds.set_format(type="torch")
    return ds

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

print("train:", len(train_ds), "| val:", len(val_ds))
print("features:", train_ds.features)

## 4. Model — regression head

In [ ]:
from transformers import AutoModelForSequenceClassification

reward_model = AutoModelForSequenceClassification.from_pretrained(
    BACKBONE_ID,
    num_labels=1,                  # regression: output 1 scalar
    ignore_mismatched_sizes=True,
)

total_params     = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
print(f"total params    : {total_params:,}")
print(f"trainable params: {trainable_params:,}")

## 5. Training

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    mse  = float(np.mean((preds - labels) ** 2))
    mae  = float(np.mean(np.abs(preds - labels)))
    # Pearson correlation
    corr = float(np.corrcoef(preds, labels)[0, 1]) if preds.std() > 0 else 0.0
    return {"mse": mse, "mae": mae, "pearson": corr}

training_args = TrainingArguments(
    output_dir=str(REWARD_MODEL_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="pearson",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=reward_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Đánh giá & lưu

In [ ]:
metrics = trainer.evaluate()
print("Val metrics:", metrics)

# Lưu best model + tokenizer
trainer.save_model(str(REWARD_MODEL_DIR))
tokenizer.save_pretrained(str(REWARD_MODEL_DIR))

# Lưu thêm engage stats để denormalize khi cần
import json
stats = {
    "engage_log_min": float(df["engage_log"].min()),
    "engage_log_max": float(df["engage_log"].max()),
    "backbone_id": BACKBONE_ID,
}
with open(REWARD_MODEL_DIR / "engage_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Đã lưu reward model tại:", REWARD_MODEL_DIR)

## 7. Sanity check — predict trên vài mẫu

In [ ]:
reward_model.eval()
reward_model.to(device)

samples = df.nlargest(3, "engage_norm")[["text", "engage_norm"]].values.tolist() + \
          df.nsmallest(3, "engage_norm")[["text", "engage_norm"]].values.tolist()

for text, true_score in samples:
    inputs = tokenizer(
        text[:200],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)
    with torch.no_grad():
        pred = reward_model(**inputs).logits.item()
    print(f"true={true_score:.3f}  pred={pred:.3f}  text={text[:60]!r}")